# Can we predict whether a student passes the closing quiz (`passed_quiz`)?
Follows `protocol.yaml`: group holdout on `student_key` (rows in `holdout.csv`), 5-fold GroupKFold CV on training rows, choice by highest mean CV ROC AUC, one holdout score for the chosen model.

**Feature timing.** The quiz closes the session, so `session_minutes` (total session length) includes the quiz itself and is only known once the outcome is determined; `hints_used` is counted over the whole session, which also covers the quiz. Neither is known before the quiz, so both are excluded. Features known before the session: `enrol_year`, `school_code`, `prior_score` (placement test). `school_code` is a code, so it is treated as categorical. `student_key` is excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest() == '5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()   # 0-based positions in data.csv
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train, hold = df[~is_hold], df[is_hold]
print(len(df), len(train), len(hold), 'student overlap:', len(set(train.student_key) & set(hold.student_key)))

In [ ]:
TARGET = 'passed_quiz'
NUM = ['enrol_year', 'prior_score']; CAT = ['school_code']
FEATURES = NUM + CAT
print(train[TARGET].mean())

Candidates are named here, before any is scored. Preprocessing lives inside each pipeline, so it is fitted within each CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), CAT)])
candidates = {
 'baseline_prior': lambda: make_pipeline(prep(False), DummyClassifier(strategy='prior')),
 'logreg_C1':      lambda: make_pipeline(prep(True), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.05':   lambda: make_pipeline(prep(True), LogisticRegression(C=0.05, max_iter=1000)),
 'rf_leaf50':      lambda: make_pipeline(prep(False), RandomForestClassifier(n_estimators=300, min_samples_leaf=50, random_state=0)),
 'hgb_small':      lambda: make_pipeline(prep(False), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0)),
}
print(list(candidates))

In [ ]:
Xtr, ytr, gtr = train[FEATURES], train[TARGET].to_numpy(), train['student_key']
folds = list(GroupKFold(n_splits=5).split(Xtr, ytr, gtr))
cv_scores = {}
for name, mk in candidates.items():
    s = []
    for tr, va in folds:
        m = mk().fit(Xtr.iloc[tr], ytr[tr])
        s.append(float(roc_auc_score(ytr[va], m.predict_proba(Xtr.iloc[va])[:, 1])))
    cv_scores[name] = s
    print(f'{name:16s} mean={np.mean(s):.4f} sd={np.std(s):.4f}')
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)

In [ ]:
final = candidates[chosen]().fit(Xtr, ytr)
hold_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:, 1]))
print(f'HEADLINE holdout roc_auc ({chosen}): {hold_auc:.4f}')

In [ ]:
train_pos = np.flatnonzero(~is_hold)
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': [[train_pos[tr].tolist(), train_pos[va].tolist()] for tr, va in folds],
 'cv_scores': cv_scores,   # candidate -> per-fold roc_auc
 'chosen': chosen,
 'holdout_rows': hold_idx.tolist(),
 'holdout_scores': {chosen: hold_auc},   # roc_auc, chosen candidate only
}
json.dump(evidence, open('evidence.json', 'w'))

## Conclusion
Only pre-session features are used, so the holdout ROC AUC is an estimate of predicting the quiz result before the session starts. The in-session variables were left out because they are recorded as the quiz outcome is determined.